In [1]:
import logging
import sys

from pathlib import Path
from rdflib import ConjunctiveGraph

sys.path.append("../..")

logging.basicConfig(level=logging.INFO, force=True)
logger = logging.getLogger()

from aggregated_traces.utils.construct_ekg import (
    check_quantities,
    generate_networkx_di_graph,
    insert_DF_DP,
    insert_fractions,
)
from aggregated_traces.utils.ekg_analysis import compute_trace_probabilities

In [2]:
from rdflib import Graph, URIRef, Variable
from typing import List


def find_lowest_quality_packing_units(g: Graph) -> List[URIRef]:
    """
    Find packing unit(s) with lowest average quality devices.
    """

    query_lowest_quality = """
    PREFIX : <http://example.org/def/ekg/aggregated_traces/>
    SELECT DISTINCT
        ?packing_unit
        (sum(?device_quality)/count(?device_quality) as ?average_quality)
    WHERE {
        [] :bizStep "packing" ;
            :parentEntity ?packing_unit ;
            :device/:quality ?device_quality .

        ?packing_unit a :PackingUnit .
    }
    GROUP BY ?packing_unit
    """

    bindings = g.query(query_lowest_quality).bindings
    min_quality = min(b[Variable("average_quality")] for b in bindings)
    return [
        b[Variable("packing_unit")]
        for b in bindings
        if b[Variable("average_quality")] == min_quality
    ]


from pandas import DataFrame, Index


def get_entity_record_number(df: DataFrame, entity: str) -> int:
    try:
        return Index(df["entity_target"]).get_loc(entity) + 1
    except KeyError:
        return None

# Process event logs

## Compute trace probabilities

In [ ]:
# Define the root cause entity
root_cause_entity = "ekg_id:DB2"

# Define folder with event log files
event_log_files = Path(
    r"C:\Users\s158699\Documents\PhD\Experiments\Simulation\aggregated_event_data\logs\scenario_1"
)

probability_dicts = {}
for event_data_file in event_log_files.glob("*.json"):
    print(event_data_file)

    # Parse data to RDF graph
    ekg_graph_rdf = ConjunctiveGraph()
    ekg_graph_rdf.parse(event_data_file)

    # Insert DF/DP relations
    ekg_graph_rdf = insert_DF_DP(ekg_graph_rdf)

    # Check incoming vs outgoing amount
    query_result_incoming_outgoing = check_quantities(ekg_graph_rdf)

    # Insert fractions on the relations
    ekg_graph_rdf = insert_fractions(ekg_graph_rdf)

    # Create NetworkX graph
    ekg_graph_nx = generate_networkx_di_graph(ekg_graph_rdf)

    # Find packing units with lowest (average) quality for which to find the root cause (entity)
    entities_source_backward = find_lowest_quality_packing_units(ekg_graph_rdf)

    # Compute backward trace probabilities
    df_backward, edges_paths_backward = compute_trace_probabilities(
        rdf_trace_graph=ekg_graph_rdf,
        nx_trace_graph=ekg_graph_nx,
        source_entities=entities_source_backward,
        trace_backward=True,
    )

    # Validate if probabilities add up to one
    if not all(
        df_backward.groupby(["entity_source", "product_model"])[["probability"]].sum()
        == 1.0
    ):
        raise Warning("Sum of probabilities by production step do not add up to one!")

    probability_dicts[event_data_file] = df_backward.to_dict(orient="records")

## Compute steps to find root cause (entity)

### Include all entities

In [6]:
results = []
for event_data_file, probability_dict in probability_dicts.items():
    df_trace = DataFrame(probability_dict)

    # Aggregate to get probabilities for target entities
    df_trace_grouped = df_trace.groupby(["entity_source", "entity_target"])[
        ["probability"]
    ].sum()
    df_trace_grouped.reset_index(inplace=True)

    entities_source = df_trace_grouped["entity_source"].unique()
    for entity_source in entities_source:
        df_selected = df_trace_grouped[
            df_trace_grouped["entity_source"] == entity_source
        ]

        # Shuffle/sample DataFrame for random order of inspection
        n_steps_random = get_entity_record_number(
            df_selected.sample(frac=1), root_cause_entity
        )

        # Sort values for informed order of inspection
        n_steps_sorted = get_entity_record_number(
            df_selected.sort_values("probability", ascending=False), root_cause_entity
        )

        results.append(
            {
                "file": event_data_file,
                "probabilities": df_selected.to_dict(orient="records"),
                "n_steps_random": n_steps_random,
                "n_steps_sorted": n_steps_sorted,
            }
        )

### Grouped by type (for example type of machine)

In [4]:
group_key = "product_model"

results = []
for event_data_file, probability_dict in probability_dicts.items():
    df_trace = DataFrame(probability_dict)

    for group_value in df_trace[group_key].unique():
        # Aggregate to get probabilities for target entities
        df_trace_grouped = (
            df_trace[df_trace[group_key] == group_value]
            .groupby(["entity_source", "entity_target"])[["probability"]]
            .sum()
        )
        df_trace_grouped.reset_index(inplace=True)

        entities_source = df_trace_grouped["entity_source"].unique()
        for entity_source in entities_source:
            df_selected = df_trace_grouped[
                df_trace_grouped["entity_source"] == entity_source
            ]

            # Shuffle/sample DataFrame for random order of inspection
            n_steps_random = get_entity_record_number(
                df_selected.sample(frac=1), root_cause_entity
            )
            # if n_steps_random: print(event_data_file, n_steps_random)

            # Sort values for informed order of inspection
            n_steps_sorted = get_entity_record_number(
                df_selected.sort_values("probability", ascending=False),
                root_cause_entity,
            )

            results.append(
                {
                    "file": event_data_file,
                    group_key: group_value,
                    "probabilities": df_selected.to_dict(orient="records"),
                    "n_steps_random": n_steps_random,
                    "n_steps_sorted": n_steps_sorted,
                }
            )

In [ ]:
df = DataFrame(results)
df.describe()

# Statistics

In [5]:
from scipy.stats import entropy

## Uniformity of probabilities

In [9]:
for i in df.index:
    df_group = DataFrame(df.loc[i, "probabilities"])

    n_targets = df_group["entity_target"].nunique()
    probabilities = df_group.groupby(["entity_source", "entity_target"])[
        "probability"
    ].sum()
    probabilities = probabilities.astype(float)

    kl_divergence = entropy(pk=probabilities.values, qk=[1 / n_targets] * n_targets)

    df.loc[i, "kl_divergence"] = (
        kl_divergence  # closer to 0 is closer to uniform distribution
    )

In [ ]:
df.describe()